# Face Recognition Model Training
## Advanced Attendance System

This notebook demonstrates how to train and evaluate the face recognition model.

In [ ]:
import sys
sys.path.append('..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

from config import Config
from models import FaceRecognitionModel
from utils import DataManager, FaceDetector

Config.create_directories()

print("Libraries imported successfully!")

## 1. Load Training Data

In [ ]:
# Load all faces
faces, labels = DataManager.load_all_faces()

print(f"Total images: {len(faces)}")
print(f"Total users: {len(set(labels))}")
print(f"\nUsers: {set(labels)}")

## 2. Visualize Sample Data

In [ ]:
# Display sample faces
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.ravel()

for i in range(min(10, len(faces))):
    axes[i].imshow(cv2.cvtColor(faces[i], cv2.COLOR_BGR2RGB))
    axes[i].set_title(labels[i])
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 3. Preprocess Data

In [ ]:
# Initialize model
model = FaceRecognitionModel()

# Prepare training data
X, y = model.prepare_training_data(faces, labels)

print(f"Feature shape: {X.shape}")
print(f"Labels shape: {y.shape}")

## 4. Split Data

In [ ]:
# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

## 5. Train KNN Model

In [ ]:
# Train KNN
model.train_knn(X_train, y_train)

# Evaluate
train_score = model.model.score(X_train, y_train)
test_score = model.model.score(X_test, y_test)

print(f"Training Accuracy: {train_score:.4f}")
print(f"Testing Accuracy: {test_score:.4f}")

## 6. Cross-Validation

In [ ]:
# Perform cross-validation
cv_scores = cross_val_score(model.model, X, y, cv=5)

print(f"Cross-validation scores: {cv_scores}")
print(f"Mean CV Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## 7. Confusion Matrix

In [ ]:
# Predictions
y_pred = model.model.predict(X_test)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

## 8. Classification Report

In [ ]:
# Get label names
label_names = model.label_encoder.inverse_transform(np.unique(y))

# Classification report
print(classification_report(y_test, y_pred, target_names=label_names))

## 9. Save Model

In [ ]:
# Save the trained model
model.save_model()
print("Model saved successfully!")

## 10. Test with Sample Image

In [ ]:
# Test prediction on a random test sample
test_idx = np.random.randint(0, len(X_test))
test_face = faces[test_idx]

# Predict
predicted_label, confidence = model.predict(test_face)
true_label = labels[test_idx]

# Display
plt.figure(figsize=(6, 6))
plt.imshow(cv2.cvtColor(test_face, cv2.COLOR_BGR2RGB))
plt.title(f"True: {true_label}\nPredicted: {predicted_label}\nConfidence: {confidence:.2f}")
plt.axis('off')
plt.show()